In [ ]:
import sys
sys.path.append("..")

from torch.utils.data import DataLoader
from wimusim.datasets import CPM

## Introduction

In this notebook, we demonstrate how to apply **parameter transformations** in WIMUSim to
introduce realistic variations in the generated virtual IMU data. This technique expands
dataset diversity without additional real-world data collection. By systematically altering
the identified WIMUSim parameters, we can simulate a broad range of conditions including
varying body morphologies, sensor placements, and hardware imperfections.

### What We Will Do in This Notebook
1. **Prepare the Identified Parameters**: Load the optimized parameters for **RealWorld**
   subjects (from `parameter_identification.ipynb` output).
2. **Define the CPM Object**: Create a `CPM` object that generates new combinations of
   B, D, P, and H parameters to reflect different subject configurations.
3. **Generate New Data**: Use CPM to generate diverse virtual IMU data for training.
4. **Personalized Dataset Generation (PDG)**: Fix some parameters to a specific subject
   while varying others for subject-specific data augmentation.

## 1. Prepare the Identified Parameters

Load the optimized parameters from `parameter_identification.ipynb` for multiple
**RealWorld** subjects. These serve as the baseline configurations for CPM.

### Understanding the Loaded Parameters
The `cpm_params` dictionary should contain:
- **`B_list`**: Body bone vectors per subject (from SMPL β).
- **`D_list`**: Dynamics (joint orientations) per subject (from SMPL pose).
- **`P_list`**: IMU placement configurations per subject.
- **`H_list`**: Hardware noise/bias models per subject.
- **`target_list`**: Real IMU signals per subject (from RealWorld dataset).

Each list has one entry per subject. Save them during parameter identification with:
```python
import pickle
cpm_params = {"B_list": [...], "D_list": [...], "P_list": [...],
              "H_list": [...], "target_list": [...]}
with open("cpm_params_realworld.pkl", "wb") as f:
    pickle.dump(cpm_params, f)
```

In [ ]:
import pickle

# Path to the CPM parameters file saved from parameter_identification.ipynb
cpm_param_path = "cpm_params_realworld.pkl"

with open(cpm_param_path, "rb") as f:
    cpm_params = pickle.load(f)

print(f"Loaded {len(cpm_params['B_list'])} subjects")

## 2. Define the Comprehensive Parameter Mixing (CPM) Object

The **CPM** object generates new combinations of B, D, P, and H parameters.
By mixing these parameter sets across subjects, we create diverse virtual IMU data
that captures realistic variability in body morphology, motion, sensor placement,
and hardware noise.

- **`window`**: Length of the sliding window for time-series segmentation.
- **`stride`**: Step size between consecutive windows.

In [ ]:
realworld_cpm_dataset = CPM(
    B_list=cpm_params["B_list"],
    D_list=cpm_params["D_list"],
    P_list=cpm_params["P_list"],
    H_list=cpm_params["H_list"],
    target_list=cpm_params["target_list"],
    window=100,
    stride=25,
    acc_only=False,
)

## **3. Generate Data with CPM**

Now that we have defined the **Comprehensive Parameter Mixing (CPM)** object, we can proceed to generate a diverse set of virtual IMU data using various combinations of the **Body (B)**, **Dynamics (D)**, **Placement (P)**, and **Hardware (H)** parameters.

### **Generating Virtual IMU Data**
The `generate_data()` method of the `CPM` object randomly choose `n_combinations` from the given parameter lists to create new virtual IMU data samples. Each sample represents a unique configuration, reflecting realistic variations in the human body model, body movement dynamics, sensor placements, and sensor characteristics.

- **`n_combinations`**: This parameter specifies the number of parameter combinations to generate. For example, setting `n_combinations=100` will produce 100 different virtual IMU samples by varying the B, D, P, and H parameters across subjects.

> **Note**: The generated data is stored in the `realdisp_cpm_dataset.data` attribute, and the corresponding labels are stored in `realdisp_cpm_dataset.target`.

In [ ]:
realworld_cpm_dataset.generate_data(
    n_combinations=10  # number of parameter combinations to generate
)

print(f"Generated {realworld_cpm_dataset.__len__()} windows of virtual IMU data.")

### **Using the Generated Data with PyTorch’s DataLoader**
The generated dataset can be used with PyTorch’s `DataLoader` to efficiently batch and shuffle the data for training machine learning models. This allows us to seamlessly integrate WIMUSim’s synthetic IMU data into model training pipelines.


### **Understanding the Output Data**
- **`data`**: Contains the generated virtual IMU signals. For each sample, the data is represented as a 3D tensor of shape `[batch_size, window_size, num_features]`.
  - `batch_size`: The number of samples in each batch (e.g., 1024).
  - `window_size`: The length of each time-series window (e.g., 100).
  - `num_features`: The number of features for each IMU signal (e.g., 12 for 6-axis IMU data with acceleration and gyroscope signals).
  
- **`target`**: Corresponding label for each sample (e.g., activity type or subject ID).
- **`idx`**: Unique identifier for each data sample, useful for tracking and debugging.


In [ ]:
data_loader = DataLoader(
    realworld_cpm_dataset,
    batch_size=1024,
    shuffle=True,
    num_workers=0,
)

for data, target, idx in data_loader:
    print(f"Data shape:   {data.shape}")
    print(f"Target shape: {target.shape}")
    print(f"Index shape:  {idx.shape}")
    break

## 4. Personalized Dataset Generation (PDG)

**PDG** fixes certain parameters (e.g. body morphology, sensor placement) for a
target subject while varying others to introduce realistic intra-subject variability.

Below we fix B, P, and H to Subject `p003` and vary only D (motion dynamics),
producing a dataset tailored to that individual's body shape and IMU placement
while covering diverse movements from all subjects.

In [ ]:
subject_idx = 2  # index into B/P/H lists corresponding to "p003"

realworld_pdg_dataset = CPM(
    B_list=cpm_params["B_list"][subject_idx : subject_idx + 1],
    D_list=cpm_params["D_list"],
    P_list=cpm_params["P_list"][subject_idx : subject_idx + 1],
    H_list=cpm_params["H_list"][subject_idx : subject_idx + 1],
    target_list=cpm_params["target_list"],
    window=100,
    stride=25,
    acc_only=False,
)

### **Generating Personalized Data**
We can now use the `generate_data()` method again to generate virtual IMU data for this subject. Since the B, P, and H parameters are fixed, only the D parameters will vary, simulating different movement patterns for Subject 3:

In [ ]:
realworld_pdg_dataset.generate_data(n_combinations=5)

print(f"Generated {realworld_pdg_dataset.__len__()} virtual IMU samples for subject p003.")

## **Conclusion**

In this notebook, we explored how to use WIMUSim's **parameter transformation capabilities** to generate diverse and realistic virtual IMU datasets for Human Activity Recognition (HAR) models. 